In [1]:
from pyspark.sql import SparkSession, Row
from pyspark.sql.types import StructType, StructField, IntegerType, FloatType, LongType, StringType
from pyspark.sql.functions import col, greatest, count, desc, least, lit, isnan, when, count, explode, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import matplotlib.ticker as mticker
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.feature import HashingTF, IDF, Tokenizer, StringIndexer
from pyspark.sql.functions import concat_ws, collect_list, lower, regexp_replace
from pyspark.ml import Pipeline
import numpy as np
from pyspark.ml.feature import Normalizer
from pyspark.ml.linalg import Vectors, SparseVector
from collections import defaultdict
import random

## Setup

In [2]:
SMALL = "../data/processed/small/"
LARGE = "../data/processed/32m/"
DATA = SMALL

## Modélisation — ALS (Alternating Least Squares)

### Principe

ALS est un algorithme de **filtrage collaboratif par factorisation matricielle**.
Il décompose la matrice sparse users × films en deux matrices denses de dimension
réduite (`rank`), dont le produit reconstruit les notes manquantes.

Les **facteurs latents** capturent automatiquement des préférences implicites
(goût pour les films sombres, préférence pour certaines époques...) sans qu'on
les définisse manuellement.

### Hyperparamètres clés

| Paramètre | Rôle | Valeur choisie |
|---|---|---|
| `rank` | Nombre de facteurs latents | 10 |
| `maxIter` | Nombre d'itérations | 10 |
| `regParam` | Régularisation (évite l'overfitting) | 0.1 |
| `coldStartStrategy` | Gestion des users/films inconnus | `drop` |

`coldStartStrategy="drop"` : si un user ou film du jeu de test n'apparaît pas
dans le train, ALS ne peut pas prédire — on supprime ces lignes plutôt que
d'obtenir `NaN` dans l'évaluation.

In [3]:
spark = SparkSession.builder \
    .appName("SparkleMovie-Modelling") \
    .master("local[*]") \
    .config("spark.driver.memory", "10g") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print(f"Spark version : {spark.version}")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/16 19:45:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/03/16 19:45:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


Spark version : 4.1.1


In [4]:
# Consistent style across all plots
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "white",
    "axes.spines.top":  False,
    "axes.spines.right":False,
    "axes.grid":        True,
    "grid.alpha":       0.3,
    "font.size":        11,
})

In [5]:
ratings_clean      = spark.read.parquet(f"{DATA}ratings_clean.parquet")
movies_clean       = spark.read.parquet(f"{DATA}movies_clean.parquet")
movies_with_genres = spark.read.parquet(f"{DATA}movies_with_genres.parquet")

print(f"ratings_clean      : {ratings_clean.count():,} rows")
print(f"movies_clean       : {movies_clean.count():,} rows")
print(f"movies_with_genres : {movies_with_genres.count():,} rows")

ratings_clean      : 100,836 rows
movies_clean       : 9,742 rows
movies_with_genres : 9,708 rows


In [6]:
train, test = ratings_clean.randomSplit([0.8, 0.2], seed=42)

print(f"Train : {train.count():,} rows")
print(f"Test  : {test.count():,} rows")

# Build ALS model
als = ALS(
    rank=10,
    maxIter=10,
    regParam=0.1,
    userCol="userId",
    itemCol="movieId",
    ratingCol="rating",
    coldStartStrategy="drop",
    seed=42
    # nonnegative=True  # Non negative factors (optional, can improve interpretability)
)

# Train
model = als.fit(train)
print("Model trained ✓")

Train : 80,578 rows
Test  : 20,258 rows
Model trained ✓


In [7]:
# Predict on test set
predictions = model.transform(test)

# RMSE
evaluator = RegressionEvaluator(
    metricName="rmse",
    labelCol="rating",
    predictionCol="prediction"
)

rmse = evaluator.evaluate(predictions)
print(f"RMSE (rank=10, maxIter=10, regParam=0.1) : {rmse:.4f}")

# Sample predictions vs actual
predictions.select("userId", "movieId", "rating", "prediction") \
            .orderBy("userId") \
            .show(10)

RMSE (rank=10, maxIter=10, regParam=0.1) : 0.8814
+------+-------+------+----------+
|userId|movieId|rating|prediction|
+------+-------+------+----------+
|     1|      6|   4.0| 4.5788817|
|     1|    943|   4.0| 3.8743846|
|     1|    101|   5.0|  4.217507|
|     1|    151|   5.0| 4.1573143|
|     1|    231|   5.0| 4.0079527|
|     1|    349|   4.0|  4.069611|
|     1|    423|   3.0| 2.7713876|
|     1|    543|   4.0| 4.2908926|
|     1|    596|   5.0| 4.2220583|
|     1|    923|   5.0|  4.381694|
+------+-------+------+----------+
only showing top 10 rows


In [8]:
# Parameter grid to explore
param_grid = (
    ParamGridBuilder()
    .addGrid(als.rank,     [5, 10, 15])
    .addGrid(als.regParam, [0.01, 0.1, 0.5])
    .build()
)

cv = CrossValidator(
    estimator=als,
    estimatorParamMaps=param_grid,
    evaluator=evaluator,
    numFolds=3,
    seed=42
)

cv_model = cv.fit(train)

# Best model params
best_model = cv_model.bestModel
print(f"Best rank     : {best_model.rank}")
print(f"Best regParam : {best_model._java_obj.parent().getRegParam()}")

# Predictions avec clipping 0.5–5.0 
best_predictions = (
    best_model.transform(test)
    .withColumn("prediction", greatest(lit(0.5), least(lit(5.0), col("prediction"))))
)

best_rmse = evaluator.evaluate(best_predictions)
print(f"Best RMSE (no leakage, clipped) : {best_rmse:.4f}")

# Vérification hors plage
out_of_range = best_predictions.filter(
    (col("prediction") < 0.5) | (col("prediction") > 5.0)
).count()
print(f"Predictions hors plage : {out_of_range}")

Best rank     : 5
Best regParam : 0.1
Best RMSE (no leakage, clipped) : 0.8748


Predictions hors plage : 0


## Résultats ALS

### Modèle initial

| Paramètre | Valeur |
|---|---|
| `rank` | 10 |
| `maxIter` | 10 |
| `regParam` | 0.1 |
| **RMSE** | **0.8814** |

Un RMSE de 0.88 signifie qu'en moyenne, la prédiction s'écarte de
0.88 étoile par rapport à la note réelle (sur une échelle de 0.5 à 5.0).

### Tuning par grid search (CrossValidator, 3 folds)

Grille explorée : `rank` ∈ {5, 10, 15} × `regParam` ∈ {0.01, 0.1, 0.5}
→ 9 combinaisons × 3 folds = 27 modèles entraînés.
Le CrossValidator est entraîné uniquement sur `train` — le test set
reste invisible pendant toute la phase de tuning.

| Paramètre | Valeur optimale |
|---|---|
| `rank` | 5 |
| `regParam` | 0.1 |
| **RMSE** | **0.8748** |

Les prédictions sont clippées entre 0.5 et 5.0 — ALS étant un modèle
de factorisation matricielle sans contrainte de bornes, il peut produire
des valeurs hors plage sans ce post-traitement.

### Interprétation

Le tuning améliore marginalement le RMSE (0.8814 → 0.8748, soit -0.007).
Un `rank=5` optimal suggère que le dataset small ne nécessite pas une
représentation complexe — 5 facteurs latents suffisent à capturer les
préférences des utilisateurs. Un `rank` trop élevé sur peu de données
risque l'overfitting, que `regParam=0.1` contribue à limiter.

In [9]:
# Pick 5 user IDs that exist in the dataset
sample_user_ids = [1, 42, 100, 200, 500]
sample_users_df = spark.createDataFrame(
    [Row(userId=uid) for uid in sample_user_ids]
)

# Get recommendations for these specific users
recs = best_model.recommendForUserSubset(sample_users_df, 10)

# Explode, clip predictions, and join with movie titles
recs_exploded = (
    recs
    .select("userId", explode("recommendations").alias("rec"))
    .select("userId", col("rec.movieId"), col("rec.rating").alias("predicted_rating"))
    .withColumn("predicted_rating", greatest(lit(0.5), least(lit(5.0), col("predicted_rating"))))
    .join(movies_clean.select("movieId", "title"), on="movieId", how="left")
    .orderBy("userId", desc("predicted_rating"))
)

recs_exploded.show(50, truncate=False)

+-------+------+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|movieId|userId|predicted_rating|title                                                                                                                        |
+-------+------+----------------+-----------------------------------------------------------------------------------------------------------------------------+
|3266   |1     |5.0             |Man Bites Dog (C'est arrivé près de chez vous) (1992)                                                                        |
|8477   |1     |5.0             |Jetée, La (1962)                                                                                                             |
|25771  |1     |5.0             |Andalusian Dog, An (Chien andalou, Un) (1929)                                                                                |
|96004  |1     |5.0             |Dragon 

## Recommandation basée sur le contenu (Content-Based)

### Principe

Contrairement à ALS qui exploite le comportement collectif des utilisateurs,
l'approche content-based analyse la **description des films** pour recommander
des contenus similaires à ceux qu'un utilisateur a appréciés.

### Pipeline

1. Construction d'un profil textuel par film (genres + tags utilisateurs)
2. Vectorisation TF-IDF : chaque film devient un vecteur numérique
3. Calcul de similarité cosinus entre films
4. Pour chaque user : identifier ses films préférés (note ≥ 4.0),
   puis recommander les films les plus similaires non encore vus

### Pourquoi TF-IDF plutôt qu'un simple comptage ?

Un genre comme "Drama" apparaît dans 40% du catalogue — il est peu
discriminant. Un tag rare comme "existentialism" ou "twist ending"
est bien plus informatif. TF-IDF pondère automatiquement les termes
rares plus fortement que les termes fréquents.

In [10]:
# Load tags
tags = spark.read.parquet(f"{DATA}tags_clean.parquet")

# Aggregate tags per movie : one row per movie with all tags concatenated
tags_per_movie = (
    tags
    .groupBy("movieId")
    .agg(concat_ws(" ", collect_list(lower(col("tag")))).alias("tags_text"))
)

# Join movies with their tags
movies_with_content = (
    movies_with_genres
    .join(tags_per_movie, on="movieId", how="left")
    .fillna("", subset=["tags_text"])
)

# Build content field : genres (pipe → space) + tags
movies_with_content = movies_with_content.withColumn(
    "content",
    concat_ws(" ",
        regexp_replace(col("genres"), "\\|", " "),
        col("tags_text")
    )
)

movies_with_content.select("movieId", "title", "content").show(5, truncate=False)

+-------+----------------------------------+-----------------------------------------------------------------------+
|movieId|title                             |content                                                                |
+-------+----------------------------------+-----------------------------------------------------------------------+
|1      |Toy Story (1995)                  |Adventure Animation Children Comedy Fantasy pixar fun                  |
|2      |Jumanji (1995)                    |Adventure Children Fantasy robin williams game fantasy magic board game|
|3      |Grumpier Old Men (1995)           |Comedy Romance old moldy                                               |
|4      |Waiting to Exhale (1995)          |Comedy Drama Romance                                                   |
|5      |Father of the Bride Part II (1995)|Comedy remake pregnancy                                                |
+-------+----------------------------------+--------------------

## TF-IDF et similarité cosinus

On transforme le champ `content` de chaque film en vecteur numérique via TF-IDF,
puis on calcule la similarité cosinus entre films pour identifier les plus proches.

In [11]:
# Step 1 : tokenize content string into list of words
tokenizer = Tokenizer(inputCol="content", outputCol="words")

# Step 2 : compute term frequency (TF)
# numFeatures=2048 : hash space size — trade-off between precision and memory
hashing_tf = HashingTF(inputCol="words", outputCol="raw_features", numFeatures=2048)

# Step 3 : compute inverse document frequency (IDF)
idf = IDF(inputCol="raw_features", outputCol="features", minDocFreq=2)

# Build and fit pipeline
pipeline = Pipeline(stages=[tokenizer, hashing_tf, idf])
tfidf_model = pipeline.fit(movies_with_content)
movies_tfidf = tfidf_model.transform(movies_with_content)

movies_tfidf.select("movieId", "title", "features").show(3, truncate=True)
print(f"TF-IDF vectors computed for {movies_tfidf.count():,} movies ✓")

+-------+--------------------+--------------------+
|movieId|               title|            features|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|(2048,[185,241,55...|
|      2|      Jumanji (1995)|(2048,[185,241,64...|
|      3|Grumpier Old Men ...|(2048,[997,1608,1...|
+-------+--------------------+--------------------+
only showing top 3 rows
TF-IDF vectors computed for 9,708 movies ✓


In [12]:
# Collect vectors to driver — acceptable for 9k movies
# For 32M dataset this would need a different approach
movies_vectors = movies_tfidf.select("movieId", "title", "features").collect()

# Build lookup dict : movieId → (title, vector)
movie_index = {
    row.movieId: (row.title, row.features)
    for row in movies_vectors
}

def cosine_similarity(v1, v2):
    """Compute cosine similarity between two sparse vectors."""
    v1_dense = np.array(v1.toArray())
    v2_dense = np.array(v2.toArray())
    norm1 = np.linalg.norm(v1_dense)
    norm2 = np.linalg.norm(v2_dense)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1_dense, v2_dense) / (norm1 * norm2))

def get_similar_movies(movie_id, top_n=10):
    """Return top_n most similar movies to a given movieId."""
    if movie_id not in movie_index:
        print(f"movieId {movie_id} not found")
        return []

    target_title, target_vector = movie_index[movie_id]
    scores = []

    for mid, (title, vector) in movie_index.items():
        if mid == movie_id:
            continue
        sim = cosine_similarity(target_vector, vector)
        scores.append((mid, title, sim))

    scores.sort(key=lambda x: x[2], reverse=True)
    return scores[:top_n]

# Test : films similaires à Toy Story (movieId=1)
print("=== Films similaires à Toy Story (1995) ===")
for mid, title, score in get_similar_movies(1, top_n=10):
    print(f"  {score:.4f}  {title}")

=== Films similaires à Toy Story (1995) ===
  0.7597  Bug's Life, A (1998)
  0.6074  Guardians of the Galaxy 2 (2017)
  0.4299  Antz (1998)
  0.4299  Adventures of Rocky and Bullwinkle, The (2000)
  0.4299  Emperor's New Groove, The (2000)
  0.4299  Monsters, Inc. (2001)
  0.4299  Wild, The (2006)
  0.4299  Shrek the Third (2007)
  0.4299  Tale of Despereaux, The (2008)
  0.4299  Asterix and the Vikings (Astérix et les Vikings) (2006)


### Recommandations content-based pour utilisateurs fictifs

Pour chaque utilisateur, on identifie ses films préférés (note ≥ 4.0),
on calcule les films les plus similaires via cosine similarity,
puis on agrège les scores en excluant les films déjà notés.

In [13]:
def recommend_content_based(user_id, top_n=10, min_rating=4.0):
    """
    Recommend movies for a user based on content similarity.
    
    Args:
        user_id   : target user
        top_n     : number of recommendations to return
        min_rating: minimum rating to consider a movie "liked"
    """
    # Get movies liked by this user
    liked = (
        ratings_clean
        .filter((col("userId") == user_id) & (col("rating") >= min_rating))
        .select("movieId", "rating")
        .collect()
    )

    if not liked:
        print(f"No ratings >= {min_rating} found for user {user_id}")
        return

    liked_ids = {row.movieId for row in liked}

    # Aggregate similarity scores across all liked movies
    scores = {}
    for row in liked:
        if row.movieId not in movie_index:
            continue
        similars = get_similar_movies(row.movieId, top_n=50)
        for mid, title, sim in similars:
            if mid in liked_ids:
                continue  # skip already seen movies
            if mid not in scores:
                scores[mid] = {"title": title, "score": 0.0, "count": 0}
            # Weight similarity by the user's rating of the source movie
            scores[mid]["score"] += sim * row.rating
            scores[mid]["count"] += 1

    # Sort by aggregated score
    ranked = sorted(scores.items(), key=lambda x: x[1]["score"], reverse=True)

    print(f"\n=== Content-based recommendations for user {user_id} ===")
    print(f"    (based on {len(liked)} liked movies)\n")
    for mid, data in ranked[:top_n]:
        print(f"  {data['score']:6.3f}  {data['title']}")

# Test on 5 users
for uid in [1, 42, 100, 200, 500]:
    recommend_content_based(uid, top_n=10)


=== Content-based recommendations for user 1 ===
    (based on 200 liked movies)

  53.789  Quest for Camelot (1998)
  52.768  Cats Don't Dance (1997)
  52.768  Many Adventures of Winnie the Pooh, The (1977)
  50.975  Rudolph, the Red-Nosed Reindeer (1964)
  50.160  Land Before Time III: The Time of the Great Giving (1995)
  50.160  Rock-A-Doodle (1991)
  50.088  Peter Pan (1953)
  50.088  Beauty and the Beast: The Enchanted Christmas (1997)
  50.088  Strange Magic (2015)
  49.776  Frosty the Snowman (1969)

=== Content-based recommendations for user 42 ===
    (based on 259 liked movies)

  87.680  Four Rooms (1995)
  87.680  Ace Ventura: When Nature Calls (1995)
  87.680  Bio-Dome (1996)
  87.680  Friday (1995)
  87.680  Black Sheep (1996)
  87.680  Mr. Wrong (1996)
  87.680  Steal Big, Steal Little (1995)
  87.680  Flirting With Disaster (1996)
  87.680  Down Periscope (1996)
  87.680  Birdcage, The (1996)

=== Content-based recommendations for user 100 ===
    (based on 107 liked 

### Résultats et limites du content-based

Le modèle produit des recommandations cohérentes avec les goûts observés :
- User 1 (200 films aimés) → films d'animation/enfants
- User 42 (259 films aimés) → comédies
- User 100 (107 films aimés) → drames romantiques

**Limite observée** : de nombreux films obtiennent des scores identiques car
ils partagent uniquement les genres sans tags distinctifs. Le TF-IDF ne peut
pas départager des films qui ont exactement le même profil de genres.

Cette limite est inhérente à la richesse des tags disponibles : seulement
3 574 tags pour 9 708 films — environ 65% des films n'ont aucun tag.
Plus le catalogue est taggé, meilleure sera la discrimination.

## Recommandation par proximité utilisateurs (KNN)

### Principe

Le KNN (K-Nearest Neighbors) identifie les utilisateurs les plus similaires
à un utilisateur cible en comparant leurs historiques de notes.

Chaque utilisateur est représenté par un vecteur sparse :
- Dimensions = films du catalogue
- Valeurs = notes données (0 si film non noté)

La similarité cosinus mesure l'angle entre deux vecteurs utilisateurs.
Deux users qui ont noté les mêmes films avec les mêmes notes auront
une similarité proche de 1.0.

### Différence clé avec ALS

ALS apprend un modèle global (facteurs latents) sur tout le dataset.
KNN ne fait aucun apprentissage — il calcule des distances à la demande.
C'est plus lent mais plus interprétable : on peut expliquer pourquoi
un film est recommandé ("parce que 5 users similaires à toi l'ont adoré").

### Hyperparamètre clé : K

K = nombre de voisins à considérer.
- K trop petit → recommandations trop personnalisées, sensibles au bruit
- K trop grand → recommandations trop génériques, perd la personnalisation
On testera K ∈ {5, 10, 20}.

In [14]:
# Build user-movie matrix as a dict : userId → {movieId: rating}
print("Building user-movie matrix...")

# Dict : userId → {movieId → rating}
user_ratings_dict = defaultdict(dict)
for row in train.select("userId", "movieId", "rating").collect():
    user_ratings_dict[row.userId][row.movieId] = row.rating

# Get all unique movie IDs and create index
all_movie_ids = sorted(movies_clean.select("movieId").rdd.flatMap(lambda x: x).collect())
movie_id_to_idx = {mid: idx for idx, mid in enumerate(all_movie_ids)}
n_movies = len(all_movie_ids)

print(f"Users  : {len(user_ratings_dict):,}")
print(f"Movies : {n_movies:,}")
print("Matrix built ✓")

Building user-movie matrix...
Users  : 610
Movies : 9,742
Matrix built ✓


In [15]:
def build_user_vector(user_id):
    """Build a sparse numpy array for a given user."""
    vec = np.zeros(n_movies)
    for movie_id, rating in user_ratings_dict[user_id].items():
        if movie_id in movie_id_to_idx:
            vec[movie_id_to_idx[movie_id]] = rating
    return vec

def cosine_sim_users(v1, v2):
    """Cosine similarity between two user vectors."""
    norm1 = np.linalg.norm(v1)
    norm2 = np.linalg.norm(v2)
    if norm1 == 0 or norm2 == 0:
        return 0.0
    return float(np.dot(v1, v2) / (norm1 * norm2))

def get_knn_recommendations(user_id, k=10, top_n=10, min_rating=4.0):
    """
    Recommend movies for a user based on K nearest neighbors.
    
    Args:
        user_id   : target user
        k         : number of neighbors to consider
        top_n     : number of recommendations to return
        min_rating: minimum rating threshold to consider neighbor's movies
    """
    if user_id not in user_ratings_dict:
        print(f"User {user_id} not found")
        return

    target_vec  = build_user_vector(user_id)
    seen_movies = set(user_ratings_dict[user_id].keys())

    # Compute similarity with all other users
    similarities = []
    for other_id in user_ratings_dict:
        if other_id == user_id:
            continue
        other_vec = build_user_vector(other_id)
        sim = cosine_sim_users(target_vec, other_vec)
        if sim > 0:
            similarities.append((other_id, sim))

    # Keep top K neighbors
    similarities.sort(key=lambda x: x[1], reverse=True)
    neighbors = similarities[:k]

    # Aggregate neighbor ratings for unseen movies
    candidate_scores = defaultdict(float)
    candidate_counts = defaultdict(int)

    for neighbor_id, sim in neighbors:
        for movie_id, rating in user_ratings_dict[neighbor_id].items():
            if movie_id in seen_movies:
                continue
            if rating < min_rating:
                continue
            candidate_scores[movie_id] += sim * rating
            candidate_counts[movie_id] += 1

    if not candidate_scores:
        print(f"No recommendations found for user {user_id}")
        return

    # Sort by aggregated score
    ranked = sorted(candidate_scores.items(), key=lambda x: x[1], reverse=True)

    # Build movie title lookup
    movie_titles = {
        row.movieId: row.title
        for row in movies_clean.select("movieId", "title").collect()
    }

    print(f"\n=== KNN (k={k}) recommendations for user {user_id} ===")
    print(f"    (based on {len(neighbors)} neighbors)\n")
    for movie_id, score in ranked[:top_n]:
        title = movie_titles.get(movie_id, "Unknown")
        print(f"  {score:6.3f}  {title}")

In [16]:
# Test with k=10 on same users as ALS and content-based
for uid in [1, 42, 100, 200, 500]:
    get_knn_recommendations(uid, k=10, top_n=10)


=== KNN (k=10) recommendations for user 1 ===
    (based on 10 neighbors)

  10.489  Twelve Monkeys (a.k.a. 12 Monkeys) (1995)
   9.858  Aliens (1986)
   9.217  Sixth Sense, The (1999)
   8.791  Terminator 2: Judgment Day (1991)
   8.567  Reservoir Dogs (1992)
   8.492  Die Hard (1988)
   8.180  Jaws (1975)
   7.707  Raising Arizona (1987)
   7.020  Blade Runner (1982)
   6.885  There's Something About Mary (1998)

=== KNN (k=10) recommendations for user 42 ===
    (based on 10 neighbors)

   9.867  American Beauty (1999)
   8.815  Sixth Sense, The (1999)
   7.967  Crouching Tiger, Hidden Dragon (Wo hu cang long) (2000)
   7.828  Pulp Fiction (1994)
   7.823  Big Lebowski, The (1998)
   7.543  Raiders of the Lost Ark (Indiana Jones and the Raiders of the Lost Ark) (1981)
   6.781  Usual Suspects, The (1995)
   6.696  Shawshank Redemption, The (1994)
   6.498  Green Mile, The (1999)
   6.408  Lord of the Rings: The Fellowship of the Ring, The (2001)

=== KNN (k=10) recommendations for 

### Résultats KNN (k=10)

Les recommandations KNN sont nettement plus diversifiées que le content-based.
On retrouve des classiques reconnus (Godfather, Blade Runner, Shawshank Redemption)
qui émergent naturellement des préférences des voisins.

**Observation — biais de popularité** : certains films très populaires
(Terminator 2, Sixth Sense) apparaissent pour plusieurs users différents.
C'est une limite inhérente au KNN : les films avec beaucoup de ratings positifs
remontent facilement car ils sont présents chez de nombreux voisins.

**Sur le temps d'exécution** : 0.3s sur le small (610 users × 9 742 films).
Sur le large (200k users), cette approche serait prohibitive sans optimisation
(indexation par LSH, réduction dimensionnelle préalable).

In [17]:
def compare_approaches(user_id, top_n=5):
    """Print side-by-side recommendations from all 3 approaches."""
    
    print(f"\n{'='*80}")
    print(f"  USER {user_id}")
    print(f"{'='*80}")
    
    # --- ALS ---
    user_df = spark.createDataFrame([Row(userId=user_id)])
    als_recs = (
        best_model.recommendForUserSubset(user_df, top_n)
        .select(explode("recommendations").alias("rec"))
        .select(col("rec.movieId"), col("rec.rating").alias("score"))
        .withColumn("score", greatest(lit(0.5), least(lit(5.0), col("score"))))
        .join(movies_clean.select("movieId", "title"), on="movieId", how="left")
        .orderBy(desc("score"))
        .collect()
    )
    
    # --- Content-based (top 5 silently) ---
    # Display function : uses full ratings_clean intentionally (not evaluation)
    liked = (
        ratings_clean
        .filter((col("userId") == user_id) & (col("rating") >= 4.0))
        .select("movieId", "rating")
        .collect()
    )
    liked_ids = {row.movieId for row in liked}
    cb_scores = {}
    for row in liked:
        if row.movieId not in movie_index:
            continue
        for mid, title, sim in get_similar_movies(row.movieId, top_n=50):
            if mid in liked_ids:
                continue
            if mid not in cb_scores:
                cb_scores[mid] = {"title": title, "score": 0.0}
            cb_scores[mid]["score"] += sim * row.rating
    cb_recs = sorted(cb_scores.items(), key=lambda x: x[1]["score"], reverse=True)[:top_n]

    # --- KNN ---
    target_vec  = build_user_vector(user_id)
    seen_movies = set(user_ratings_dict[user_id].keys())
    similarities = []
    for other_id in user_ratings_dict:
        if other_id == user_id:
            continue
        sim = cosine_sim_users(target_vec, build_user_vector(other_id))
        if sim > 0:
            similarities.append((other_id, sim))
    similarities.sort(key=lambda x: x[1], reverse=True)
    neighbors = similarities[:10]
    knn_scores = defaultdict(float)
    for neighbor_id, sim in neighbors:
        for movie_id, rating in user_ratings_dict[neighbor_id].items():
            if movie_id not in seen_movies and rating >= 4.0:
                knn_scores[movie_id] += sim * rating
    movie_titles = {r.movieId: r.title for r in movies_clean.select("movieId", "title").collect()}
    knn_recs = sorted(knn_scores.items(), key=lambda x: x[1], reverse=True)[:top_n]

    # --- Print side by side ---
    print(f"\n  {'ALS':30} {'Content-Based':40} {'KNN':30}")
    print(f"  {'-'*28} {'-'*38} {'-'*28}")
    
    for i in range(top_n):
        als_title = als_recs[i].title[:28] if i < len(als_recs) else ""
        cb_title  = cb_recs[i][1]["title"][:38] if i < len(cb_recs) else ""
        knn_title = movie_titles.get(knn_recs[i][0], "")[:28] if i < len(knn_recs) else ""
        print(f"  {als_title:30} {cb_title:40} {knn_title:30}")

for uid in [1, 42, 100, 200, 500]:
    compare_approaches(uid, top_n=5)


  USER 1

  ALS                            Content-Based                            KNN                           
  ---------------------------- -------------------------------------- ----------------------------
  Man Bites Dog (C'est arrivé    Quest for Camelot (1998)                 Twelve Monkeys (a.k.a. 12 Mo  
  Jetée, La (1962)               Cats Don't Dance (1997)                  Aliens (1986)                 
  Andalusian Dog, An (Chien an   Many Adventures of Winnie the Pooh, Th   Sixth Sense, The (1999)       
  Dragon Ball Z: The History o   Rudolph, the Red-Nosed Reindeer (1964)   Terminator 2: Judgment Day (  
  On the Beach (1959)            Land Before Time III: The Time of the    Reservoir Dogs (1992)         

  USER 42

  ALS                            Content-Based                            KNN                           
  ---------------------------- -------------------------------------- ----------------------------
  Stranger Than Paradise (1984   Four Rooms 

## Interprétation

ALS recommande des films très de niche — La Jetée (1962), Andalusian Dog, Dragon Ball Z. Ce sont des films peu connus avec des facteurs latents très spécifiques. C'est la signature du filtrage collaboratif matriciel : il capte des patterns subtils mais peut sembler bizarre.

Content-based est cohérent mais limité — user 42 et 500 ont exactement les mêmes recommandations (Four Rooms, Ace Ventura, Bio-Dome). C'est le problème des scores identiques qu'on a vu — pas assez de tags pour discriminer.

KNN donne les résultats les plus "humainement compréhensibles" — des classiques reconnus, bien notés. Mais avec le biais popularité qu'on a identifié.

## Phase 3 — Évaluation et comparaison des approches

### Métriques utilisées

Pour comparer les trois approches de manière rigoureuse, on utilise :

**RMSE (Root Mean Square Error)** — pour ALS uniquement
Mesure l'écart entre notes prédites et réelles. Ne s'applique pas au
content-based et KNN qui ne prédisent pas de notes explicites.

**Precision@K**
Sur les K films recommandés, quelle proportion l'utilisateur aurait
réellement aimée (note ≥ 4.0 dans le test set) ?
```
Precision@K = |films recommandés ∩ films aimés dans le test| / K
```

**Coverage**
Quel pourcentage du catalogue l'algorithme est capable de recommander ?
Un algo qui recommande toujours les mêmes 100 films populaires a une
couverture faible.

In [18]:
# Split already done : train / test
# We evaluate on the 5 sample users for consistency

def precision_at_k_als(user_ids, k=10, min_rating=4.0):
    """Compute Precision@K for ALS on a list of users."""
    precisions = []

    for user_id in user_ids:
        # Ground truth : movies liked in test set
        liked_test = set(
            test
            .filter((col("userId") == user_id) & (col("rating") >= min_rating))
            .select("movieId")
            .rdd.flatMap(lambda x: x)
            .collect()
        )

        if not liked_test:
            continue

        # ALS recommendations
        user_df = spark.createDataFrame([Row(userId=user_id)])
        recs = (
            best_model.recommendForUserSubset(user_df, k)
            .select(explode("recommendations").alias("rec"))
            .select(col("rec.movieId"))
            .rdd.flatMap(lambda x: x)
            .collect()
        )

        hits = len(set(recs) & liked_test)
        precisions.append(hits / k)

    return np.mean(precisions) if precisions else 0.0

sample_users = [1, 42, 100, 200, 500]
p_at_10_als = precision_at_k_als(sample_users, k=10)
print(f"ALS     Precision@10 : {p_at_10_als:.4f}")

ALS     Precision@10 : 0.0200


In [19]:
def precision_at_k_knn(user_ids, k=10, min_rating=4.0):
    """Compute Precision@K for KNN."""
    precisions = []

    for user_id in user_ids:
        liked_test = set(
            test
            .filter((col("userId") == user_id) & (col("rating") >= min_rating))
            .select("movieId")
            .rdd.flatMap(lambda x: x)
            .collect()
        )
        if not liked_test:
            continue

        # KNN recs
        target_vec  = build_user_vector(user_id)
        seen_movies = set(user_ratings_dict[user_id].keys())
        similarities = sorted(
            [(oid, cosine_sim_users(target_vec, build_user_vector(oid)))
             for oid in user_ratings_dict if oid != user_id],
            key=lambda x: x[1], reverse=True
        )[:10]

        knn_scores = defaultdict(float)
        for neighbor_id, sim in similarities:
            for movie_id, rating in user_ratings_dict[neighbor_id].items():
                if movie_id not in seen_movies and rating >= min_rating:
                    knn_scores[movie_id] += sim * rating

        recs = [mid for mid, _ in sorted(
            knn_scores.items(), key=lambda x: x[1], reverse=True
        )[:k]]

        hits = len(set(recs) & liked_test)
        precisions.append(hits / k)

    return np.mean(precisions) if precisions else 0.0


def precision_at_k_cb(user_ids, k=10, min_rating=4.0):
    precisions = []
    for user_id in user_ids:
        liked_test = set(
            test
            .filter((col("userId") == user_id) & (col("rating") >= min_rating))
            .select("movieId")
            .rdd.flatMap(lambda x: x)
            .collect()
        )
        if not liked_test:
            continue

        # ← train uniquement, pas ratings_clean
        liked_train = (
            train
            .filter((col("userId") == user_id) & (col("rating") >= min_rating))
            .select("movieId", "rating")
            .collect()
        )
        liked_ids = {r.movieId for r in liked_train}

        cb_scores = {}
        for row in liked_train:
            if row.movieId not in movie_index:
                continue
            for mid, title, sim in get_similar_movies(row.movieId, top_n=50):
                if mid in liked_ids:
                    continue
                if mid not in cb_scores:
                    cb_scores[mid] = 0.0
                cb_scores[mid] += sim * row.rating

        recs = [mid for mid, _ in sorted(
            cb_scores.items(), key=lambda x: x[1], reverse=True
        )[:k]]

        hits = len(set(recs) & liked_test)
        precisions.append(hits / k)

    return np.mean(precisions) if precisions else 0.0

p_at_10_knn = precision_at_k_knn(sample_users, k=10)
p_at_10_cb  = precision_at_k_cb(sample_users,  k=10)

print(f"ALS          Precision@10 : {p_at_10_als:.4f}")
print(f"KNN          Precision@10 : {p_at_10_knn:.4f}")
print(f"Content-based Precision@10 : {p_at_10_cb:.4f}")

ALS          Precision@10 : 0.0200
KNN          Precision@10 : 0.2000
Content-based Precision@10 : 0.0400


In [20]:
def compute_coverage(recs_list, catalog_size):
    """Proportion of catalog covered by recommendations."""
    all_recommended = set()
    for recs in recs_list:
        all_recommended.update(recs)
    return len(all_recommended) / catalog_size

In [ ]:
random.seed(42)
all_users = list(user_ratings_dict.keys())
eval_users = random.sample(all_users, min(50, len(all_users)))

catalog_size = movies_clean.count()
als_recs_all, knn_recs_all, cb_recs_all = [], [], []

movie_titles = {r.movieId: r.title for r in movies_clean.select("movieId", "title").collect()}

for user_id in eval_users:
    # ALS
    user_df = spark.createDataFrame([Row(userId=user_id)])
    als_recs_all.append(set(
        best_model.recommendForUserSubset(user_df, 10)
        .select(explode("recommendations").alias("rec"))
        .select(col("rec.movieId"))
        .rdd.flatMap(lambda x: x).collect()
    ))

    # KNN
    target_vec  = build_user_vector(user_id)
    seen_movies = set(user_ratings_dict[user_id].keys())
    sims = sorted(
        [(oid, cosine_sim_users(target_vec, build_user_vector(oid)))
         for oid in user_ratings_dict if oid != user_id],
        key=lambda x: x[1], reverse=True
    )[:10]
    knn_s = defaultdict(float)
    for nid, sim in sims:
        for mid, rating in user_ratings_dict[nid].items():
            if mid not in seen_movies and rating >= 4.0:
                knn_s[mid] += sim * rating
    knn_recs_all.append(set(sorted(knn_s, key=knn_s.get, reverse=True)[:10]))

    # Content-based
    liked = train.filter(
        (col("userId") == user_id) & (col("rating") >= 4.0)
    ).select("movieId", "rating").collect()
    liked_ids = {r.movieId for r in liked}
    cb_s = {}
    for row in liked:
        if row.movieId not in movie_index:
            continue
        for mid, title, sim in get_similar_movies(row.movieId, top_n=50):
            if mid not in liked_ids:
                cb_s[mid] = cb_s.get(mid, 0.0) + sim * row.rating
    cb_recs_all.append(set(sorted(cb_s, key=cb_s.get, reverse=True)[:10]))

cov_als = compute_coverage(als_recs_all, catalog_size)
cov_knn = compute_coverage(knn_recs_all, catalog_size)
cov_cb  = compute_coverage(cb_recs_all,  catalog_size)

print(f"\n=== Coverage (50 users, top-10 recs) ===")
print(f"ALS           : {cov_als:.4f} ({cov_als*100:.1f}%)")
print(f"KNN           : {cov_knn:.4f} ({cov_knn*100:.1f}%)")
print(f"Content-based : {cov_cb:.4f}  ({cov_cb*100:.1f}%)")


=== Coverage (50 users, top-10 recs) ===
ALS           : 0.0125 (1.3%)
KNN           : 0.0180 (1.8%)
Content-based : 0.0298  (3.0%)


26/03/16 21:25:37 ERROR Inbox: Ignoring error
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:53)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRefByURI(RpcEnv.scala:102)
	at org.apache.spark.rpc.RpcEnv.setupEndpointRef(RpcEnv.scala:110)
	at org.apache.spark.util.RpcUtils$.makeDriverRef(RpcUtils.scala:36)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.driverEndpoint$lzycompute(BlockManagerMasterEndpoint.scala:132)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.org$apache$spark$storage$BlockManagerMasterEndpoint$$driverEndpoint(BlockManagerMasterEndpoint.scala:131)
	at org.apache.spark.storage.BlockManagerMasterEndpoint.isExecutorAlive$lzycompute$1(BlockManagerMasterEndpoint.scala:707)
	at org.apache.spark.storage.BlockManagerMasterE

Precision@10 = 0.20 pour KNN signifie : sur 10 films recommandés, en moyenne 2 se retrouvent dans les films que l'user a réellement aimés dans le test set. C'est faible en valeur absolue, mais voici le contexte :
Le test set d'un user contient en moyenne ~20 films (20% de son historique). Le catalogue total fait 9 742 films. Si on recommandait 10 films au hasard, la probabilité de tomber sur un film du test set serait de 20/9742 ≈ 0.002 soit 0.2%. KNN atteint 20% — c'est 100x mieux que le hasard. C'est donc un bon signal.

ApprocheForceFaiblesseKNNMeilleure précision (20%)Biais popularité, lent sur largeContent-basedMeilleure couverture (3%)Discrimination faible sans tagsALSRecommandations originales/nichePrécision faible sur ce dataset

## Conclusion — Comparaison des trois approches

### Tableau récapitulatif des métriques

| Approche | RMSE | Precision@10 | Coverage (50 users) | Complexité |
|---|---|---|---|---|
| **ALS** | **0.8748** | 2.0% | 1.3% | Haute (factorisation matricielle) |
| **KNN** | — | **20.0%** | 1.8% | Moyenne (calcul de distances) |
| **Content-based** | — | 4.0% | **3.0%** | Faible (similarité cosinus) |

> Le RMSE n'est calculé que pour ALS — seul algorithme produisant des prédictions
> de notes explicites. KNN et Content-based produisent des rankings, pas des notes.

---

### Interprétation des résultats

**ALS — Filtrage collaboratif matriciel**

ALS obtient un RMSE de 0.8748 après tuning, ce qui signifie qu'en moyenne
ses prédictions de notes s'écartent de moins d'une étoile de la réalité.
En revanche, sa Precision@10 de 2% révèle une faiblesse : le modèle tend à
recommander des films très de niche (facteurs latents très spécifiques) qui
ne correspondent pas aux films effectivement appréciés dans le test set.

Ce comportement est caractéristique d'ALS sur un petit dataset : avec seulement
610 utilisateurs, les facteurs latents capturent des corrélations très fines
qui ne se généralisent pas bien. Sur le dataset large (200k users), ALS serait
attendu bien plus performant.

**KNN — Proximité utilisateurs**

KNN obtient la meilleure Precision@10 avec 20%, soit 100 fois mieux qu'une
recommandation aléatoire (baseline théorique ≈ 0.2%). Ses recommandations
sont intuitivement cohérentes — des classiques bien notés émergent naturellement
des préférences des voisins.

Sa principale limite est le **biais de popularité** : les films très populaires
(Terminator 2, Sixth Sense) remontent pour plusieurs utilisateurs différents,
ce qui réduit la personnalisation. Par ailleurs, cette approche est difficilement
scalable : sur 200 000 utilisateurs, calculer toutes les similarités paires serait
prohibitif sans optimisation (LSH, ANN indexing).

**Content-based — TF-IDF et similarité cosinus**

Le content-based obtient la meilleure couverture catalogue (3%) — il peut
recommander n'importe quel film ayant un genre, indépendamment de son historique
de ratings. C'est un avantage crucial pour les films récents ou peu populaires
(cold start problem).

Sa Precision@10 de 4% reste modeste, ce qui s'explique par la faible richesse
des tags : seulement 3 574 tags pour 9 708 films (65% des films sans aucun tag).
La discrimination repose alors uniquement sur les genres, qui sont trop grossiers
pour différencier des films similaires.

---

### Limites de l'évaluation

Plusieurs biais méthodologiques doivent être mentionnés :

- La **Precision@10 est calculée sur 5 utilisateurs seulement** — un échantillon
  trop petit pour être statistiquement représentatif. Les résultats sont indicatifs.
- Le **dataset small (100k ratings)** favorise KNN par rapport à ALS : ALS montre
  sa vraie puissance sur des volumes plus importants (millions de ratings).
- Le **biais de sélection** de MovieLens (utilisateurs actifs, cinéphiles engagés)
  rend difficile la généralisation à un public grand public.

---

### Recommandation finale

Aucune approche n'est universellement supérieure — chacune répond à un besoin
différent. En production, un **système hybride** combinant les trois approches
serait plus robuste :
```
score_final = α × score_ALS + β × score_KNN + γ × score_content_based
```

- **ALS** pour la personnalisation profonde sur utilisateurs actifs
- **KNN** pour les utilisateurs avec un historique de ratings dense
- **Content-based** pour les nouveaux utilisateurs et les films récents
  (résout le cold start problem)

Le choix des coefficients α, β, γ dépendrait des objectifs métier :
maximiser la précision, la diversité, ou la couverture catalogue.